# Zadanie 3 Extra — DeepWeeds: wiarygodność ewaluacji

**Pytanie badawcze:** czy wynik modelu na opublikowanym podziale DeepWeeds odzwierciedla rozpoznawanie gatunków chwastów, czy może jest częściowo zawyżony przez bardzo podobne ujęcia obecne w zbiorach treningowym i testowym?

Twoim celem nie jest pobicie modelu bazowego. Masz zaprojektować i przeprowadzić badanie, które pozwoli ocenić wiarygodność jego ewaluacji. Równie wartościowy jest wynik potwierdzający hipotezę, jak i poprawnie uzasadniony wynik, który jej nie potwierdza.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from huggingface_hub import hf_hub_download, snapshot_download
from PIL import Image
from safetensors.torch import load_file
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

In [ ]:
DATA_DIR = Path("data/zadanie-3")
MODEL_DIR = Path("models/zadanie-3-model-bazowy")
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

images_path = hf_hub_download(
    repo_id="AI-MED-AGH/Recruitment-Task-3",
    filename="original/images.zip",
    repo_type="dataset",
    local_dir=DATA_DIR,
)
dataset_snapshot = snapshot_download(
    repo_id="AI-MED-AGH/Recruitment-Task-3",
    repo_type="dataset",
    allow_patterns=["labels/*.csv"],
    local_dir=DATA_DIR,
)
model_snapshot = snapshot_download(
    repo_id="AI-MED-AGH/Recruitment-Task-3-Model",
    local_dir=MODEL_DIR,
)
with ZipFile(images_path) as archive:
    archive.extractall(DATA_DIR / "images")

print(f"Dane: {dataset_snapshot}")
print(f"Model bazowy: {model_snapshot}")

## Najważniejsze zasady

- Korzystaj z opublikowanego **foldu 0**: `train_subset0.csv`, `val_subset0.csv` i `test_subset0.csv`. We wszystkich porównaniach zachowaj ten sam zbiór testowy.
- Sposób mierzenia podobieństwa i jego próg ustal na danych treningowych i walidacyjnych, zanim przeanalizujesz wyniki testowe.
- Główną metryką jest **macro-F1**. Pokaż również wyniki dla poszczególnych klas.
- We wszystkich wariantach użyj tego samego modelu, preprocessingu i procedury treningowej.
- Zadbaj o odtwarzalność: zapisz najważniejsze ustawienia, ziarno losowe i wersje bibliotek.

## A — Hipoteza i plan badania

Przeczytaj oryginalną publikację DeepWeeds oraz co najmniej jedno dodatkowe źródło dotyczące duplikatów lub zależnych obserwacji w ewaluacji modeli. Krótko wyjaśnij, dlaczego podobne ujęcia w treningu i teście mogą zawyżać wynik.

Zapisz jedną hipotezę: usunięcie z treningu obrazów bardzo podobnych do testowych obniży wynik bardziej niż usunięcie takiej samej liczby losowych obrazów. Przed eksperymentem określ, jak duża różnica będzie dla Ciebie istotna praktycznie.

Uruchom dostarczony checkpoint, aby sprawdzić wczytywanie danych i inferencję. Jest to celowo słaby model techniczny. Do właściwego eksperymentu wybierz jeden rozsądny model na podstawie zbiorów treningowego i walidacyjnego.

## B — Eksperyment

Wybierz i uzasadnij jeden sposób wykrywania podobnych obrazów, na przykład perceptual hash albo odległość embeddingów. Ustal próg na danych treningowych i walidacyjnych, a następnie ręcznie sprawdź przykładowe pary uznane za podobne i niepodobne.

Sprawdź, ile obrazów testowych ma bardzo podobny odpowiednik w zbiorze treningowym i jak zjawisko rozkłada się między klasami. Następnie przygotuj trzy warianty zbioru treningowego:

1. **oryginalny** — pełny trening z foldu 0;
2. **oczyszczony** — bez obrazów treningowych podobnych do testowych;
3. **kontrolny** — bez takiej samej liczby losowych obrazów, z zachowaniem podobnych liczebności klas.

Wytrenuj ten sam model w każdym wariancie i oceń wszystkie wersje na niezmienionym zbiorze testowym. Jeżeli pełny trening jest zbyt kosztowny, możesz użyć lekkiego modelu albo zamrożonych cech — jednakowo we wszystkich trzech wariantach.

## C — Analiza dowodów i wniosek

Porównaj macro-F1 i wyniki klas dla trzech wariantów. Najważniejsza jest różnica między wariantem oczyszczonym i kontrolnym, ponieważ oba zawierają mniej danych. Dodaj prostą ocenę niepewności, na przykład przedział ufności albo powtórzenie eksperymentu dla kilku ziaren.

Pokaż przykłady podobnych par i reprezentatywne błędy. Sprawdź również, czy model częściej poprawnie klasyfikuje obrazy testowe mające podobny odpowiednik w treningu. Pamiętaj, że taki związek sam w sobie nie dowodzi zapamiętywania obrazów.

Zakończ odpowiedzią na pytanie badawcze i odnieś ją do ustalonego wcześniej kryterium. Wynik niepotwierdzający hipotezy jest poprawnym wynikiem. Wskaż najważniejsze ograniczenia badania, w tym fakt, że eksperyment dotyczy jednego foldu i nie zastępuje testu na danych z nowego miejsca lub czasu.

## Oddanie zadania

Zapisz w notatniku hipotezę, kod, wyniki, wykresy i wnioski. Dodaj krótką tabelę porównującą trzy warianty oraz opisz ograniczenia badania. Podaj wykorzystane źródła. Opublikuj kod w repozytorium.